# Projeto G1 — Tema 3
# Análise de Queimadas no Brasil

 Linguagens de Programação
 Professor: Alexandre Neves Louzada
 Aluno: Luis Fernando Dantas Carvalho

## 1. Introdução

Este notebook realiza a análise exploratória da base `simulacao_queimadas_brasil.csv`, conforme o projeto G1 — Tema 3.

O objetivo é investigar a evolução dos focos de queimadas entre 2015 e 2024, identificar estados e regiões mais afetados, analisar biomas e sazonalidade, e verificar a relação entre índice de seca e quantidade de focos.

> **Observação:** a base é simulada e deve ser tratada como exercício acadêmico.

## 2. Contextualização ambiental

As queimadas podem afetar biodiversidade, qualidade do ar, saúde pública, mudanças climáticas, recursos naturais e produção agrícola.

A análise de dados permite transformar os registros em indicadores, gráficos e evidências para apoiar a identificação de períodos e áreas críticas.

## 3. Explicação da base

A base contém as seguintes variáveis:

- `ano`: ano da ocorrência;
- `mes`: mês;
- `data`: data de referência;
- `regiao`: região brasileira;
- `uf`: estado;
- `bioma`: bioma predominante;
- `focos_queimada`: quantidade de focos;
- `temperatura_media`: temperatura média;
- `chuva_mm`: volume de chuva;
- `area_atingida_km2`: área atingida;
- `indice_seca`: índice de estiagem;
- `qualidade_ar`: índice de qualidade do ar;
- `nivel_risco`: nível de risco.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

df = pd.read_csv("../dados/simulacao_queimadas_brasil.csv")
df.head()

,ano,mes,data,regiao,uf,bioma,focos_queimada,temperatura_media,chuva_mm,area_atingida_km2,indice_seca,qualidade_ar,nivel_risco
0,2015,1,2015-01-01,Norte,AM,Cerrado,43,29.6,40.9,29.62,0.46,65.6,Alto
1,2015,1,2015-01-01,Norte,PA,Cerrado,49,30.3,27.7,5.25,0.71,60.8,Alto
2,2015,1,2015-01-01,Norte,RO,Pampa,19,31.6,43.5,7.35,0.56,84.8,Médio
3,2015,1,2015-01-01,Norte,TO,Pampa,16,26.9,139.0,7.17,0.46,87.2,Médio
4,2015,1,2015-01-01,Nordeste,BA,Mata Atlântica,6,28.4,105.9,3.07,0.14,95.2,Baixo


## 4. Leitura e inspeção dos dados

In [2]:
print("Dimensões:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

print("\nTipos:")
display(df.dtypes)

print("\nValores nulos:")
display(df.isna().sum())

print("\nDuplicatas:", df.duplicated().sum())

Dimensões: (2400, 13)

Colunas:
['ano', 'mes', 'data', 'regiao', 'uf', 'bioma', 'focos_queimada', 'temperatura_media', 'chuva_mm', 'area_atingida_km2', 'indice_seca', 'qualidade_ar', 'nivel_risco']

Tipos:


ano                    int64
mes                    int64
data                     str
regiao                   str
uf                       str
bioma                    str
focos_queimada         int64
temperatura_media    float64
chuva_mm             float64
area_atingida_km2    float64
indice_seca          float64
qualidade_ar         float64
nivel_risco              str
dtype: object


Valores nulos:


ano                  0
mes                  0
data                 0
regiao               0
uf                   0
bioma                0
focos_queimada       0
temperatura_media    0
chuva_mm             0
area_atingida_km2    0
indice_seca          0
qualidade_ar         0
nivel_risco          0
dtype: int64


Duplicatas: 0


## 5. Limpeza e preparação

In [3]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["ano"] = pd.to_numeric(df["ano"], errors="coerce").astype("Int64")
df["mes"] = pd.to_numeric(df["mes"], errors="coerce").astype("Int64")

colunas_numericas = [
    "focos_queimada", "temperatura_media", "chuva_mm",
    "area_atingida_km2", "indice_seca", "qualidade_ar"
]

for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remoção de linhas inválidas para a análise
df = df.dropna().copy()

print("Dimensões após preparação:", df.shape)
print("Nulos restantes:", df.isna().sum().sum())

Dimensões após preparação: (2400, 13)
Nulos restantes: 0


## 6. Engenharia de atributos

In [4]:
mes_nome = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}

df["mes_nome"] = df["mes"].map(mes_nome)

# Classificação simples para apoiar a leitura do índice de seca
df["faixa_seca"] = pd.cut(
    df["indice_seca"],
    bins=[-0.01, 0.33, 0.66, 1.01],
    labels=["Baixa", "Média", "Alta"]
)

df[["ano", "mes", "mes_nome", "indice_seca", "faixa_seca"]].head()

,ano,mes,mes_nome,indice_seca,faixa_seca
0,2015,1,Janeiro,0.46,Média
1,2015,1,Janeiro,0.71,Alta
2,2015,1,Janeiro,0.56,Média
3,2015,1,Janeiro,0.46,Média
4,2015,1,Janeiro,0.14,Baixa


## 7. KPIs

In [5]:
total_focos = df["focos_queimada"].sum()
area_total = df["area_atingida_km2"].sum()
estado_mais_afetado = df.groupby("uf")["focos_queimada"].sum().idxmax()
regiao_mais_critica = df.groupby("regiao")["focos_queimada"].sum().idxmax()
mes_mais_critico = int(df.groupby("mes")["focos_queimada"].sum().idxmax())
media_anual = df.groupby("ano")["focos_queimada"].sum().mean()

print(f"Total de focos: {total_focos:,.0f}")
print(f"Área total atingida: {area_total:,.2f} km²")
print(f"Estado mais afetado: {estado_mais_afetado}")
print(f"Região mais crítica: {regiao_mais_critica}")
print(f"Mês mais crítico: {mes_mais_critico}")
print(f"Média anual de queimadas: {media_anual:,.1f}")

Total de focos: 61,803
Área total atingida: 24,874.64 km²
Estado mais afetado: GO
Região mais crítica: Nordeste
Mês mais crítico: 10
Média anual de queimadas: 6,180.3


## 8. Evolução temporal

In [6]:
temporal = df.groupby("data", as_index=False)["focos_queimada"].sum()

fig = px.line(
    temporal,
    x="data",
    y="focos_queimada",
    markers=True,
    title="Evolução mensal dos focos de queimadas"
)
fig.update_layout(xaxis_title="Data", yaxis_title="Focos")
fig.show()

anual = df.groupby("ano", as_index=False)["focos_queimada"].sum()
display(anual)

,ano,focos_queimada
0,2015,5420
1,2016,5552
2,2017,5584
3,2018,5779
4,2019,5660
5,2020,6171
6,2021,6617
7,2022,6796
8,2023,7166
9,2024,7058


## 9. Comparação entre regiões

In [7]:
regioes = (
    df.groupby("regiao", as_index=False)["focos_queimada"]
      .sum()
      .sort_values("focos_queimada", ascending=False)
)

fig = px.bar(
    regioes,
    x="regiao",
    y="focos_queimada",
    text_auto=True,
    title="Focos de queimadas por região"
)
fig.show()
regioes

,regiao,focos_queimada
1,Nordeste,15078
3,Sudeste,12564
2,Norte,12560
0,Centro-Oeste,12517
4,Sul,9084


## 10. Comparação entre estados e ranking crítico

In [8]:
ranking_estados = (
    df.groupby("uf", as_index=False)["focos_queimada"]
      .sum()
      .sort_values("focos_queimada", ascending=False)
)

fig = px.bar(
    ranking_estados,
    x="uf",
    y="focos_queimada",
    text_auto=True,
    title="Ranking de estados por focos"
)
fig.show()

display(ranking_estados)

,uf,focos_queimada
5,GO,3379
15,RO,3325
10,PA,3231
14,RJ,3227
4,ES,3191
16,RS,3146
18,SP,3119
11,PB,3119
12,PE,3077
2,CE,3070


## 11. Análise por bioma

In [9]:
biomas = (
    df.groupby("bioma", as_index=False)["focos_queimada"]
      .sum()
      .sort_values("focos_queimada", ascending=False)
)

fig = px.bar(
    biomas,
    x="bioma",
    y="focos_queimada",
    text_auto=True,
    title="Focos por bioma"
)
fig.show()

display(biomas)

,bioma,focos_queimada
5,Pantanal,13225
0,Amazônia,12857
2,Cerrado,12522
3,Mata Atlântica,8247
4,Pampa,8164
1,Caatinga,6788


## 12. Análise sazonal e heatmap

In [10]:
sazonal = df.groupby("mes", as_index=False)["focos_queimada"].sum()

fig = px.bar(
    sazonal,
    x="mes",
    y="focos_queimada",
    text_auto=True,
    title="Sazonalidade mensal dos focos"
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

heat = df.pivot_table(
    index="mes",
    columns="ano",
    values="focos_queimada",
    aggfunc="sum",
    fill_value=0
)

fig = px.imshow(
    heat,
    aspect="auto",
    text_auto=True,
    labels=dict(x="Ano", y="Mês", color="Focos"),
    title="Heatmap mensal de queimadas"
)
fig.show()

## 13. Relação entre seca e queimadas

In [11]:
corr = df[["indice_seca", "focos_queimada"]].corr().iloc[0, 1]
print(f"Correlação de Pearson: {corr:.3f}")

fig = px.scatter(
    df,
    x="indice_seca",
    y="focos_queimada",
    color="nivel_risco",
    hover_data=["ano", "mes", "uf", "regiao", "bioma"],
    trendline="ols",
    title="Índice de seca × focos de queimadas"
)
fig.update_layout(
    xaxis_title="Índice de seca",
    yaxis_title="Focos de queimadas"
)
fig.show()

Correlação de Pearson: 0.815


### Interpretação

A correlação mede associação linear, não causalidade. Nesta base simulada, o coeficiente é positivo e forte, indicando que registros com maior índice de seca tendem a apresentar mais focos.

Essa evidência pode ser utilizada para justificar maior atenção preventiva em períodos de estiagem.

## 14. Nível de risco

In [12]:
risco = (
    df.groupby("nivel_risco", as_index=False)["focos_queimada"]
      .sum()
      .sort_values("focos_queimada", ascending=False)
)

display(risco)

fig = px.bar(
    risco,
    x="nivel_risco",
    y="focos_queimada",
    text_auto=True,
    title="Focos por nível de risco"
)
fig.show()

,nivel_risco,focos_queimada
0,Alto,33199
3,Médio,23629
2,Crítico,3148
1,Baixo,1827


## 15. Tabela dinâmica

In [13]:
tabela_dinamica = pd.pivot_table(
    df,
    index="uf",
    columns="ano",
    values="focos_queimada",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

display(tabela_dinamica)

ano,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,Total
uf,,,,,,,,,,,
AM,213,239,344,323,240,373,311,287,364,307,3001
BA,249,227,316,305,329,309,253,276,368,230,2862
CE,305,256,278,321,272,285,328,338,354,333,3070
DF,237,290,278,312,282,342,269,418,306,334,3068
ES,295,237,300,241,288,286,358,396,401,389,3191
GO,314,348,256,322,303,260,304,397,467,408,3379
MA,253,365,275,251,230,302,355,292,254,373,2950
MG,293,333,255,232,325,259,280,306,454,290,3027
MS,307,210,280,290,245,264,375,369,336,371,3047


## 16. Interpretação dos resultados

A base completa apresenta **61.803 focos** e **24.874,64 km²** de área atingida.

O estado com maior quantidade acumulada de focos é **GO**, enquanto a região com maior total é o **Nordeste**. O mês de maior ocorrência é **outubro** e o bioma com maior quantidade de focos é o **Pantanal**.

A série anual apresenta crescimento do total de focos de 2015 para 2024, embora existam oscilações entre anos. A análise sazonal mostra concentração principalmente nos meses de julho a outubro.

A relação entre índice de seca e focos é positiva e forte na base simulada, reforçando a utilidade de indicadores climáticos para análise preventiva.

Como o dataset é simulado, essas conclusões não devem ser interpretadas como estatísticas oficiais do Brasil.

## 17. Conclusão

O projeto demonstra como técnicas de análise e visualização podem apoiar a investigação de queimadas. Os KPIs, rankings, séries temporais, heatmap, análise por bioma e correlação entre seca e focos permitem identificar padrões e períodos de maior atenção.

O dashboard desenvolvido em Streamlit transforma essas análises em uma aplicação interativa, permitindo que o usuário aplique filtros e explore diferentes recortes da base.